# 03 — Experimentos y evaluación final

Se comparan 12 configuraciones: Dummy, regresión logística, SVM, Random Forest y
XGBoost. Los dos últimos están sugeridos por el profesor. Gana el F1 macro medio
en validación cruzada; test no decide el ganador.

Este notebook muestra la ejecución guardada por `python -m src.models.train`.
Si no existe, la ejecuta. Para reproducir desde cero use ese comando; no cambie
hiperparámetros después de leer test para luego presentar ese mismo test como nuevo.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Abra Jupyter desde la raíz del proyecto")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data.dataset import load_data, split_data

df = load_data()
X_train, X_test, y_train, y_test = split_data(df)
print("Entrenamiento:", X_train.shape, "Test reservado:", X_test.shape)

In [ ]:
import json

from src.models.train import train

results = ROOT / "docs/results"
if not (results / "evaluation.json").exists():
    train()
evaluation = json.loads((results / "evaluation.json").read_text(encoding="utf-8"))
display(pd.read_csv(results / "leaderboard.csv"))
print("Modelo seleccionado por CV:", evaluation["winner"])
print("Parámetros:", evaluation["best_params"])

## Test reservado
Son 400 ejemplos que no participaron en el ajuste. Las métricas describen este dataset, no una validación comercial externa.

In [ ]:
display(pd.Series(evaluation["metrics"], name="valor"))
display(pd.DataFrame(evaluation["classification_report"]).T)
print("Cumple metas académicas propuestas:", evaluation["acceptance_passed"])
from IPython.display import Image, display

display(Image(filename=str(results / "confusion_matrix.png")))

## Tracking y registro

La ejecución guarda parámetros, métricas y artefactos en MLflow local (`mlflow.db`).
Para abrirlo, desde la raíz ejecute:

```text
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

Abra http://127.0.0.1:5000. En el experimento `mobile-price-classification` verá
una ejecución por algoritmo y el CSV con todas sus configuraciones. El mejor pipeline
se registra como `mobile-price-classifier`; recibe alias `candidate`; `make promote` ejecuta el gate antes de asignar `champion`.
La base y los modelos permanecen locales, mientras las métricas se publican en GitHub.

**Siguiente etapa:** explicar los errores, orquestar con Prefect y servir el pipeline
completo. No afirmar que Random Forest o XGBoost son mejores sin leer los resultados.
